# Notebook 04 — Stage 2: Edge Cases, Performance & Scalability

**FDIC Bank–Quarter Project | Part 3 — Scalable Processing [25 pts]**

This notebook is independent from Stage 1 and **does not download FDIC APIs or modify Part 1/2 Parquet**. It reproduces the exact 03B cleaning policy used in Stage 1, tests unusual cases, then benchmarks the same three logical workloads using **Polars Lazy**, **pandas**, and **PySpark local**.

**Staged execution**: start with `RUN_MODE='PILOT'` (approximately 25% of the 1,682,616 rows, chronological year prefix). Review parity and benchmark files first. Later change to `RUN_MODE='FULL'` and `CONFIRM_FULL=True` to run 25%/50%/75%/100% cumulative datasets. The experiment **does not claim a real distributed cluster**.

**Workloads**:

- **W1** — Scan/filter/project `CERT, REPDTE, ASSET`, compute count + asset total.
- **W2** — Load source fields, compute 13 `*_QA_VALID` fields, three QA counters, and materialized summaries of all QA-valid fields.
- **W3** — W2 + calendar broadcast/small-dimension join + bank reporting-gap window + quarter-level aggregation.

All three engines materialize comparable small output tables before timing stops. Each workload is warmed up. For Polars and pandas, subprocess isolation avoids one engine's allocations contaminating the other's memory measurements; Spark runs in a separate local worker process with JVM RSS included.
**Patch v2 (2026-11-15):** In `polars_query` W1, `is_between` date bounds use `pl.lit(...)` to prevent `ColumnNotFoundError` for e.g. `19900101`.


## 0. Install & embed reproducible worker

Run this cell once in Colab. The helper module is embedded below, so **only this `.ipynb` is needed**. Google Colab must be connected to the internet to install dependencies. Python version and installed dependency versions are recorded at export.

In [ ]:
%pip -q install "polars>=1.32,<2" "pyarrow>=17,<25" "pandas>=2.2,<3" "pyspark>=3.5,<4.1" "psutil>=5.9" "numpy>=1.26,<3"

In [ ]:
import os, sys, json, time, math, shutil, zipfile, subprocess, tempfile, platform
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
import pyarrow as pa
import psutil
import importlib.util

LOCAL_ROOT = Path('/content/fdic_part3_stage2') if Path('/content').exists() else (Path(tempfile.gettempdir())/'fdic_part3_stage2')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
WORKER_PATH = LOCAL_ROOT / 'fdic_part3_stage2_worker.py'

In [ ]:
WORKER_SOURCE = r'''
"""Self-contained Stage 2 worker. It is embedded into the companion notebook.
Never writes to source Parquet or clean-panel folders.
"""
import os
import sys
import json
import gc
import time
import argparse
import threading
import statistics
from pathlib import Path
import psutil
import numpy as np
import pandas as pd

BALANCE_NONNEGATIVE_FIELDS = [
    'ASSET','DEP','DEPDOM','COREDEP','DEPINS','DEPUNINS','CHBAL',
    'SC','LNLSGR','LNLSNET','P3ASSET','P9ASSET','NAASSET'
]
RATIO_REVIEW_ABS_THRESHOLDS = {
    'ROA': 100.0, 'ROE': 1000.0, 'NIMY': 100.0, 'RBC1AAJ': 1000.0
}
DENOMINATOR_REVIEW_FIELDS = ['ASSET','DEP','LNLSGR']
KEYS = ['CERT','REPDTE']
RAW_KEEP = list(dict.fromkeys(
    KEYS + BALANCE_NONNEGATIVE_FIELDS + list(RATIO_REVIEW_ABS_THRESHOLDS)
    + ['EQ','NETINC']
))
VALID_FIELDS = [f'{c}_QA_VALID' for c in BALANCE_NONNEGATIVE_FIELDS]
QA_FIELDS = [
    'QA_NEGATIVE_BALANCE_COUNT', 'QA_RATIO_REVIEW_COUNT', 'QA_ZERO_DENOM_COUNT'
]
W1, W2, W3 = 'W1_FILTER', 'W2_CLEAN_QA', 'W3_END_TO_END'
WORKLOADS = (W1, W2, W3)


def calendar_for_years(years):
    periods = pd.period_range(f'{min(years)}Q1', f'{max(years)}Q4', freq='Q')
    return pd.DataFrame([
        {'REPDTE': p.end_time.strftime('%Y%m%d'),
         'CAL_YEAR': int(p.year), 'CAL_QUARTER': int(p.quarter)}
        for p in periods
    ])


def pandas_core(raw):
    df = raw[RAW_KEEP].copy()
    negative_counts = np.zeros(len(df), dtype=np.int32)
    ratio_counts = np.zeros(len(df), dtype=np.int32)
    denom_counts = np.zeros(len(df), dtype=np.int32)
    for c in BALANCE_NONNEGATIVE_FIELDS:
        invalid = df[c].lt(0).fillna(False)
        df[f'{c}_QA_VALID'] = df[c].mask(invalid)
        negative_counts += invalid.to_numpy(dtype=np.int32)
    for c, lim in RATIO_REVIEW_ABS_THRESHOLDS.items():
        ratio_counts += df[c].abs().gt(lim).fillna(False).to_numpy(dtype=np.int32)
    for c in DENOMINATOR_REVIEW_FIELDS:
        denom_counts += df[c].eq(0).fillna(False).to_numpy(dtype=np.int32)
    df['QA_NEGATIVE_BALANCE_COUNT'] = negative_counts
    df['QA_RATIO_REVIEW_COUNT'] = ratio_counts
    df['QA_ZERO_DENOM_COUNT'] = denom_counts
    return df


def pandas_final(core, calendar):
    df = core.merge(calendar, on='REPDTE', how='left', validate='many_to_one')
    assert df['CAL_YEAR'].notna().all(), 'Unmatched reporting date'
    df['Q_ORD'] = df['CAL_YEAR'] * 4 + df['CAL_QUARTER'] - 1
    df = df.sort_values(KEYS, kind='mergesort').reset_index(drop=True)
    df['PREV_Q_ORD'] = df.groupby('CERT', sort=False)['Q_ORD'].shift(1)
    df['GAP_QUARTERS'] = (df['Q_ORD'] - df['PREV_Q_ORD'] - 1).clip(lower=0)
    return df


def pandas_calc(raw, calendar, workload, last_year):
    if workload == W1:
        x = raw.loc[
            raw['REPDTE'].between(f'{last_year}0101', f'{last_year}1231'),
            ['CERT','REPDTE','ASSET']
        ]
        return pd.DataFrame([{'ROWS': len(x), 'ASSET_TOTAL': float(x['ASSET'].sum())}])
    core = pandas_core(raw)
    if workload == W2:
        out = {'ROWS': len(core)}
        for c in QA_FIELDS:
            out[c] = int(core[c].sum())
        for c in VALID_FIELDS:
            out[f'{c}_SUM'] = float(core[c].sum(skipna=True))
            out[f'{c}_NULLS'] = int(core[c].isna().sum())
        return pd.DataFrame([out])
    if workload == W3:
        fin = pandas_final(core, calendar)
        out = fin.assign(_HAS_GAP=(fin['GAP_QUARTERS'] > 0).astype('int64'))
        agg = out.groupby('REPDTE', as_index=False).agg(
            BANK_COUNT=('CERT','size'),
            ASSET_TOTAL=('ASSET','sum'),
            NEG_BALANCE_FLAGS=('QA_NEGATIVE_BALANCE_COUNT','sum'),
            REPORTING_GAP_BANKS=('_HAS_GAP','sum'),
        )
        return agg.sort_values('REPDTE').reset_index(drop=True)
    raise ValueError(workload)


def polars_core(raw_lf):
    import polars as pl
    base = raw_lf.select(RAW_KEEP)
    neg_flags = [(pl.col(c) < 0).fill_null(False).cast(pl.Int32)
                 for c in BALANCE_NONNEGATIVE_FIELDS]
    ratio_flags = [(pl.col(c).abs() > lim).fill_null(False).cast(pl.Int32)
                   for c, lim in RATIO_REVIEW_ABS_THRESHOLDS.items()]
    denom_flags = [(pl.col(c) == 0).fill_null(False).cast(pl.Int32)
                   for c in DENOMINATOR_REVIEW_FIELDS]
    return base.with_columns([
        *[pl.when(pl.col(c) < 0).then(None).otherwise(pl.col(c))
          .alias(f'{c}_QA_VALID') for c in BALANCE_NONNEGATIVE_FIELDS],
        pl.sum_horizontal(neg_flags).cast(pl.Int32).alias(QA_FIELDS[0]),
        pl.sum_horizontal(ratio_flags).cast(pl.Int32).alias(QA_FIELDS[1]),
        pl.sum_horizontal(denom_flags).cast(pl.Int32).alias(QA_FIELDS[2]),
    ])


def polars_final(core_lf, calendar):
    import polars as pl
    x = core_lf.join(pl.from_pandas(calendar).lazy(), on='REPDTE', how='left', validate='m:1')
    x = x.with_columns(
        (pl.col('CAL_YEAR') * 4 + pl.col('CAL_QUARTER') - 1)
        .cast(pl.Int32).alias('Q_ORD')
    )
    x = x.sort(KEYS).with_columns(
        pl.col('Q_ORD').shift(1).over('CERT').alias('PREV_Q_ORD')
    )
    return x.with_columns(
        pl.when(pl.col('PREV_Q_ORD').is_null()).then(None)
        .otherwise((pl.col('Q_ORD') - pl.col('PREV_Q_ORD') - 1).clip(lower_bound=0))
        .cast(pl.Int32).alias('GAP_QUARTERS')
    )


def polars_query(raw_lf, calendar, workload, last_year):
    import polars as pl
    if workload == W1:
        return (raw_lf.filter(pl.col('REPDTE').is_between(
            pl.lit(f'{last_year}0101'), pl.lit(f'{last_year}1231')))
            .select(['CERT','REPDTE','ASSET'])
            .select([pl.len().alias('ROWS'),
                     pl.col('ASSET').sum().alias('ASSET_TOTAL')]))
    core = polars_core(raw_lf)
    if workload == W2:
        exprs = [pl.len().alias('ROWS')]
        exprs += [pl.col(c).sum().alias(c) for c in QA_FIELDS]
        for c in VALID_FIELDS:
            exprs.append(pl.col(c).sum().fill_null(0).alias(f'{c}_SUM'))
            exprs.append(pl.col(c).null_count().alias(f'{c}_NULLS'))
        return core.select(exprs)
    if workload == W3:
        fin = polars_final(core, calendar)
        return (fin.group_by('REPDTE').agg([
            pl.len().alias('BANK_COUNT'),
            pl.col('ASSET').sum().alias('ASSET_TOTAL'),
            pl.col(QA_FIELDS[0]).sum().alias('NEG_BALANCE_FLAGS'),
            (pl.col('GAP_QUARTERS') > 0).fill_null(False).cast(pl.Int64)
            .sum().alias('REPORTING_GAP_BANKS'),
        ]).sort('REPDTE'))
    raise ValueError(workload)


def spark_core(sdf):
    from pyspark.sql import functions as F
    s = sdf.select(*RAW_KEEP)
    neg, ratio, denom = [], [], []
    for c in BALANCE_NONNEGATIVE_FIELDS:
        bad = F.col(c) < 0
        s = s.withColumn(f'{c}_QA_VALID',
                         F.when(bad, F.lit(None)).otherwise(F.col(c)))
        neg.append(F.when(bad, F.lit(1)).otherwise(F.lit(0)))
    for c, lim in RATIO_REVIEW_ABS_THRESHOLDS.items():
        ratio.append(F.when(F.abs(F.col(c)) > lim, 1).otherwise(0))
    for c in DENOMINATOR_REVIEW_FIELDS:
        denom.append(F.when(F.col(c) == 0, 1).otherwise(0))
    def sum_expr(parts):
        result = F.lit(0)
        for x in parts:
            result = result + x
        return result
    return (s.withColumn(QA_FIELDS[0], sum_expr(neg))
             .withColumn(QA_FIELDS[1], sum_expr(ratio))
             .withColumn(QA_FIELDS[2], sum_expr(denom)))


def spark_final(core, calendar, spark):
    from pyspark.sql import functions as F
    from pyspark.sql import Window
    cal = spark.createDataFrame(
        [(str(r.REPDTE), int(r.CAL_YEAR), int(r.CAL_QUARTER))
         for r in calendar.itertuples(index=False)],
        schema='REPDTE string, CAL_YEAR int, CAL_QUARTER int'
    )
    joined = (core.join(F.broadcast(cal), on='REPDTE', how='left')
              .withColumn('Q_ORD',
                          (F.col('CAL_YEAR')*4 + F.col('CAL_QUARTER') - 1).cast('int')))
    window = Window.partitionBy('CERT').orderBy('REPDTE')
    return (joined.withColumn('PREV_Q_ORD', F.lag('Q_ORD').over(window))
            .withColumn('GAP_QUARTERS',
                F.when(F.col('PREV_Q_ORD').isNull(), F.lit(None).cast('int'))
                 .otherwise(F.greatest(F.col('Q_ORD')-F.col('PREV_Q_ORD')-1, F.lit(0)))))


def spark_query(sdf, calendar, workload, last_year, spark):
    from pyspark.sql import functions as F
    if workload == W1:
        return (sdf.filter(F.col('REPDTE').between(
            f'{last_year}0101', f'{last_year}1231'))
            .select('CERT','REPDTE','ASSET')
            .agg(F.count('*').alias('ROWS'), F.sum('ASSET').alias('ASSET_TOTAL')))
    core = spark_core(sdf)
    if workload == W2:
        ex = [F.count('*').alias('ROWS')]
        ex += [F.sum(F.col(c)).alias(c) for c in QA_FIELDS]
        for c in VALID_FIELDS:
            ex.append(F.coalesce(F.sum(F.col(c)), F.lit(0)).alias(f'{c}_SUM'))
            ex.append(F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f'{c}_NULLS'))
        return core.agg(*ex)
    if workload == W3:
        fin = spark_final(core, calendar, spark)
        return (fin.groupBy('REPDTE').agg(
            F.count('*').alias('BANK_COUNT'),
            F.sum('ASSET').alias('ASSET_TOTAL'),
            F.sum(QA_FIELDS[0]).alias('NEG_BALANCE_FLAGS'),
            F.sum(F.when(F.col('GAP_QUARTERS') > 0, 1).otherwise(0))
            .alias('REPORTING_GAP_BANKS')))
    raise ValueError(workload)


def records_from_pandas(frame):
    # to_json normalizes numpy scalars and nullable pandas values without
    # transmitting whole source tables (outputs here have at most ~170 rows).
    return json.loads(frame.to_json(orient='records', date_format='iso', double_precision=14))


def records_from_polars(frame):
    return frame.to_dicts()


def records_from_spark(frame):
    return [row.asDict(recursive=True) for row in frame.collect()]


def rss_tree_mib():
    proc = psutil.Process(os.getpid())
    total = 0
    try:
        for p in [proc] + proc.children(recursive=True):
            try:
                total += p.memory_info().rss
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                pass
    except psutil.NoSuchProcess:
        pass
    return total / 1024**2


def sample_call(call, interval=0.04):
    """Wall time and sampled total RSS including Spark JVM child process."""
    gc.collect()
    baseline = rss_tree_mib()
    peak = [baseline]
    done = threading.Event()
    def watch():
        while not done.wait(interval):
            try:
                peak[0] = max(peak[0], rss_tree_mib())
            except Exception:
                pass
    watcher = threading.Thread(target=watch, daemon=True)
    watcher.start()
    start = time.perf_counter()
    try:
        output = call()
    finally:
        elapsed = (time.perf_counter() - start)*1000
        done.set()
        watcher.join(timeout=2)
        peak[0] = max(peak[0], rss_tree_mib())
    return output, elapsed, baseline, peak[0]


def run_worker(cfg):
    engine = cfg['engine']
    files = list(cfg['files'])
    years = list(cfg['years'])
    last_year = max(years)
    calendar = calendar_for_years(years)
    repeats = int(cfg['repeats'])
    outdir = Path(cfg['outdir'])
    outdir.mkdir(parents=True, exist_ok=True)
    workload_names = list(cfg.get('workloads', WORKLOADS))
    import pyarrow.parquet as pq
    expected_rows = sum(int(pq.ParquetFile(p).metadata.num_rows) for p in files)
    print(f'Worker {engine} | {expected_rows:,} rows | {len(files)} files', flush=True)
    spark = None
    if engine == 'spark':
        from pyspark.sql import SparkSession
        spark = (SparkSession.builder.master('local[2]')
                .appName('FDIC_Part3_Stage2_Local')
                .config('spark.sql.shuffle.partitions','8')
                .config('spark.sql.adaptive.enabled','true')
                .getOrCreate())
        spark.sparkContext.setLogLevel('ERROR')

    output_rows, record_outputs = [], {}
    plans = {}
    try:
        for workload in workload_names:
            print(f'  [{engine}] {workload}: warm-up', flush=True)
            def task():
                if engine == 'pandas':
                    cols = ['CERT','REPDTE','ASSET'] if workload == W1 else RAW_KEEP
                    raw = pd.concat([
                        pd.read_parquet(p, columns=cols, engine='pyarrow')
                        for p in files
                    ], ignore_index=True)
                    return records_from_pandas(pandas_calc(raw, calendar, workload, last_year))
                if engine == 'polars':
                    import polars as pl
                    lf = pl.scan_parquet(files, hive_partitioning=False)
                    query = polars_query(lf, calendar, workload, last_year)
                    if workload not in plans:
                        plans[workload] = query.explain(optimized=True)
                    return records_from_polars(query.collect())
                if engine == 'spark':
                    # Spark DataFrame is a lazy plan. collect() forces each timed read.
                    cols = ['CERT','REPDTE','ASSET'] if workload == W1 else RAW_KEEP
                    sdf = spark.read.parquet(*files).select(*cols)
                    query = spark_query(sdf, calendar, workload, last_year, spark)
                    rows = records_from_spark(query)
                    if workload not in plans:
                        plans[workload] = query._jdf.queryExecution().executedPlan().toString()
                    return rows
                raise ValueError(engine)
            _ = task()  # warm-up is deliberately not measured
            repeats_rows = []
            for rep in range(1, repeats+1):
                result, elapsed, baseline, peak = sample_call(task)
                if rep == 1:
                    record_outputs[workload] = result
                repeats_rows.append({
                    'engine':engine, 'workload':workload,
                    'repeat':rep, 'elapsed_ms':round(elapsed,3),
                    'rss_before_mib':round(baseline,3),
                    'peak_rss_mib':round(peak,3),
                    'delta_peak_rss_mib':round(max(0, peak-baseline),3),
                    'rows_input':int(expected_rows),
                    'year_start':int(min(years)), 'year_end':int(max(years)),
                })
                print(f'  [{engine}] {workload} repeat {rep}/{repeats} '
                      f'{elapsed:.1f} ms | RSS peak {peak:.1f} MiB', flush=True)
            output_rows.extend(repeats_rows)
        data = {
            'engine':engine, 'years':years, 'input_rows':int(expected_rows),
            'measurements':output_rows, 'results':record_outputs,
            'plans':plans,
            'memory_metric':'sampled worker+children RSS (40ms polling); '
                            'includes JVM for Spark; not allocation profiler',
        }
        (outdir/f'{engine}_result.json').write_text(
            json.dumps(data,indent=2,ensure_ascii=False),encoding='utf-8')
        for workload,plan in plans.items():
            (outdir/f'{engine}_{workload}_plan.txt').write_text(plan,encoding='utf-8')
        return data
    finally:
        if spark is not None:
            spark.stop()


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    args=parser.parse_args()
    cfg=json.loads(Path(args.config).read_text(encoding='utf-8'))
    run_worker(cfg)


if __name__ == '__main__':
    main()

'''
WORKER_PATH.write_text(WORKER_SOURCE, encoding='utf-8')
import py_compile
py_compile.compile(str(WORKER_PATH), doraise=True)
if str(LOCAL_ROOT) not in sys.path: sys.path.insert(0, str(LOCAL_ROOT))
import fdic_part3_stage2_worker as logic
print('Embedded worker:', WORKER_PATH)
print('Polars', pl.__version__, '| pandas', pd.__version__, '| PyArrow', pa.__version__)

Embedded worker: /content/fdic_part3_stage2/fdic_part3_stage2_worker.py
Polars 1.35.2 | pandas 2.2.3 | PyArrow 23.0.1


## 1. Edge-case tests — **run before touching FULL data**

Synthetic tests reproduce Stage 1 formulas **without using real-bank identifiers**:

1. Two negatives on one row → `QA_NEGATIVE_BALANCE_COUNT = 2`; original financial columns unchanged.
2. All three denominator candidates equal zero → counter = 3.
3. Null balance is **not** interpreted as negative and `*_QA_VALID` remains null.
4. Extreme ROA / ROE values count toward the review flag, but are **not capped or removed**.
5. Missing intermediate quarter within one year and **across a year boundary** → `GAP_QUARTERS = 1`; first observation has null gap.
6. Compare row-level outputs **Polars–pandas–Spark**; an error stops the notebook instead of silently passing.

Spark uses `local[2]`, so this is a functional local-engine test rather than proof of a cluster.

In [ ]:
TEST_SPARK_EDGE = True   # Stage 1 PySpark passed; keep True for the complete edge-case test.

base = {c: 20.0 for c in logic.RAW_KEEP if c not in ('CERT','REPDTE')}
base.update({'ASSET':100.,'DEP':70.,'LNLSGR':60.,'SC':10.,'ROA':1.,'ROE':5.,
             'NIMY':2.,'RBC1AAJ':10.,'EQ':15.,'NETINC':3.})

def event_row(cert, repdte, **overrides):
    d={'CERT':int(cert),'REPDTE':repdte,**base}
    d.update(overrides)
    return d

synthetic = pd.DataFrame([
    event_row(101,'20190331'),
    event_row(101,'20190930',SC=-7.,LNLSGR=-2.,ROA=120.),
    event_row(101,'20191231',ASSET=0.,DEP=0.,LNLSGR=0.),
    event_row(202,'20190630',SC=None,EQ=-10.),
    event_row(202,'20190930',ROE=-2000.),
    event_row(303,'20191231'),
    event_row(303,'20200630'),
])[logic.RAW_KEEP]
cal = logic.calendar_for_years([2019,2020])

edge_pd = logic.pandas_final(logic.pandas_core(synthetic),cal)
edge_pl = logic.polars_final(
    logic.polars_core(pl.from_pandas(synthetic).lazy()),cal
).collect().to_pandas()

COMPARE_COLS=logic.VALID_FIELDS+logic.QA_FIELDS+['CAL_YEAR','CAL_QUARTER','Q_ORD','PREV_Q_ORD','GAP_QUARTERS']
EDGE_CHECKS=[]

def edge_check(name, ok, note=''):
    EDGE_CHECKS.append({'test':name,'status':'PASS' if bool(ok) else 'FAIL','note':note})
    print(('PASS' if bool(ok) else 'FAIL'),name,flush=True)

def compare_row_level(left,right,label):
    a=left.sort_values(logic.KEYS).reset_index(drop=True)
    b=right.sort_values(logic.KEYS).reset_index(drop=True)
    edge_check(label+' | keys',len(a)==len(b) and a[logic.KEYS].equals(b[logic.KEYS]))
    for c in COMPARE_COLS:
        x=pd.to_numeric(a[c],errors='coerce').to_numpy(dtype='float64',na_value=np.nan)
        y=pd.to_numeric(b[c],errors='coerce').to_numpy(dtype='float64',na_value=np.nan)
        edge_check(f'{label} | {c}', bool(np.isclose(x,y,rtol=1e-8,atol=1e-6,equal_nan=True).all()))

compare_row_level(edge_pl,edge_pd,'Polars-pandas')

def value_at(frame,cert,q,col):
    subset=frame.loc[(frame.CERT==cert)&(frame.REPDTE==q),col]
    assert len(subset)==1,(cert,q,col)
    return subset.iloc[0]

edge_check('Two negatives flagged',value_at(edge_pd,101,'20190930',logic.QA_FIELDS[0])==2)
edge_check('Three zero denominators',value_at(edge_pd,101,'20191231',logic.QA_FIELDS[2])==3)
edge_check('Null SC preserved',pd.isna(value_at(edge_pd,202,'20190630','SC_QA_VALID')))
edge_check('Negative original preserved',value_at(edge_pd,101,'20190930','SC')==-7.)
edge_check('Extreme ROA review',value_at(edge_pd,101,'20190930',logic.QA_FIELDS[1])==1)
edge_check('Extreme negative ROE review',value_at(edge_pd,202,'20190930',logic.QA_FIELDS[1])==1)
edge_check('Within-year gap=1',value_at(edge_pd,101,'20190930','GAP_QUARTERS')==1)
edge_check('Cross-year gap=1',value_at(edge_pd,303,'20200630','GAP_QUARTERS')==1)
edge_check('First quarter gap null',pd.isna(value_at(edge_pd,101,'20190331','GAP_QUARTERS')))

if TEST_SPARK_EDGE:
    from pyspark.sql import SparkSession
    from pyspark.sql.types import StructType,StructField,LongType,StringType,DoubleType
    spark_edge=(SparkSession.builder.master('local[2]')
        .appName('FDIC_Part3_EdgeCase_Only')
        .config('spark.sql.shuffle.partitions','4')
        .getOrCreate())
    spark_edge.sparkContext.setLogLevel('ERROR')
    spark_schema=StructType([
        StructField('CERT',LongType(),False),
        StructField('REPDTE',StringType(),False),
        *[StructField(c,DoubleType(),True) for c in logic.RAW_KEEP if c not in logic.KEYS]
    ])
    numeric_cols=[c for c in logic.RAW_KEEP if c not in logic.KEYS]
    records=[(int(r.CERT),str(r.REPDTE),
              *[(None if pd.isna(getattr(r,c)) else float(getattr(r,c))) for c in numeric_cols])
             for r in synthetic.itertuples(index=False)]
    sdf=spark_edge.createDataFrame(records,schema=spark_schema)
    edge_spark=logic.spark_final(logic.spark_core(sdf),cal,spark_edge).toPandas()
    compare_row_level(edge_pl,edge_spark,'Polars-Spark')
    spark_edge.stop()
else:
    print('Spark edge-case parity skipped; do not claim three-engine validation.')

edge_df=pd.DataFrame(EDGE_CHECKS)
edge_df.to_csv(LOCAL_ROOT/'stage2_edge_case_checks.csv',index=False)
assert (edge_df.status=='PASS').all(), 'EDGE CASE FAILURE: inspect the failed checks before benchmarking.'
assert TEST_SPARK_EDGE, 'Enable Spark edge tests before publishing the three-engine comparison.'
print('EDGE CASE TESTS:',len(edge_df),'/',len(edge_df),'PASS')

PASS Polars-pandas | keys
PASS Polars-pandas | ASSET_QA_VALID
PASS Polars-pandas | DEP_QA_VALID
PASS Polars-pandas | DEPDOM_QA_VALID
PASS Polars-pandas | COREDEP_QA_VALID
PASS Polars-pandas | DEPINS_QA_VALID
PASS Polars-pandas | DEPUNINS_QA_VALID
PASS Polars-pandas | CHBAL_QA_VALID
PASS Polars-pandas | SC_QA_VALID
PASS Polars-pandas | LNLSGR_QA_VALID
PASS Polars-pandas | LNLSNET_QA_VALID
PASS Polars-pandas | P3ASSET_QA_VALID
PASS Polars-pandas | P9ASSET_QA_VALID
PASS Polars-pandas | NAASSET_QA_VALID
PASS Polars-pandas | QA_NEGATIVE_BALANCE_COUNT
PASS Polars-pandas | QA_RATIO_REVIEW_COUNT
PASS Polars-pandas | QA_ZERO_DENOM_COUNT
PASS Polars-pandas | CAL_YEAR
PASS Polars-pandas | CAL_QUARTER
PASS Polars-pandas | Q_ORD
PASS Polars-pandas | PREV_Q_ORD
PASS Polars-pandas | GAP_QUARTERS
PASS Two negatives flagged
PASS Three zero denominators
PASS Null SC preserved
PASS Negative original preserved
PASS Extreme ROA review
PASS Extreme negative ROE review
PASS Within-year gap=1
PASS Cross-year 

## 2. Run configuration, source files & stage gates

- `RUN_MODE='PILOT'` executes **only S25** (a chronological prefix that first reaches ~25% of total rows).
- `RUN_MODE='FULL'` and `CONFIRM_FULL=True` executes **S25, S50, S75, S100**. Change only after pilot parity passes.
- Each size includes **complete years**, not randomly sampled bank-quarters. Exact row fractions are printed.
- Every timed engine uses **local `/content` copies** of the *same* immutable source Parquet files.
- pandas and Polars use separate process-isolated workers, each with warm-up and repeated runs. Spark is local, also isolated, and measured separately.
- Peak memory is **sampled process-tree RSS**, not an exact allocator peak. Spark values include its JVM; do not interpret them as pure DataFrame memory.

In [ ]:
RUN_MODE = "FULL"
CONFIRM_FULL = True
RUN_SPARK_BENCHMARK = True  # Do not skip Spark if comparing 3 engines.
SPARK_FULL_SCALES = ('S25','S100')  # Spark local on two representative sizes.
PILOT_REPEATS = 3
FULL_REPEATS = 5
SPARK_PILOT_REPEATS = 2
SPARK_FULL_REPEATS = 3
TOTAL_EXPECTED_ROWS = 1_682_616
LOCAL_STAGE_MAX_DISK_FRACTION = 0.70

assert RUN_MODE in ('PILOT','FULL')
if RUN_MODE=='FULL':
    assert CONFIRM_FULL, 'FULL safety gate: set CONFIRM_FULL=True after pilot PASS.'

root_override=os.environ.get('FDIC_PROJECT_ROOT')
if root_override:
    PROJECT_ROOT=Path(root_override).expanduser().resolve()
else:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT=Path('/content/drive/MyDrive/FDIC_Bank_Failure_Project')
FIN_DIR=PROJECT_ROOT/'data_storage/financials/year_partitioned_full'
REPORT_DIR=PROJECT_ROOT/'outputs/part3/stage2_benchmark_v1'/RUN_MODE.lower()
REPORT_DIR.mkdir(parents=True,exist_ok=True)
STAGE1_REPORT=PROJECT_ROOT/'outputs/part3/stage1_pilot_v1/stage1_acceptance_gates.csv'
assert STAGE1_REPORT.is_file(), f'Stage1 report missing: {STAGE1_REPORT}'
checks=pd.read_csv(STAGE1_REPORT)
assert (checks.status=='PASS').all(), 'Stage 1 has non-PASS gates. Review before Stage 2.'
assert FIN_DIR.is_dir(),FIN_DIR
if RUN_MODE=='FULL':
    prior=PROJECT_ROOT/'outputs/part3/stage2_benchmark_v1/pilot/stage2_acceptance_gates.csv'
    assert prior.is_file(), f'Run Stage 2 PILOT first; missing {prior}'
    pilot_checks=pd.read_csv(prior)
    assert (pilot_checks.status=='PASS').all(), 'Previous PILOT report is not all PASS.'
assert str(REPORT_DIR.resolve()).startswith(str((PROJECT_ROOT/'outputs/part3').resolve()))
print('Stage 1:',len(checks),'PASS | Project:',PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Stage 1: 7 PASS | Project: /content/drive/MyDrive/FDIC_Bank_Failure_Project


In [ ]:
# Source metadata only: no full Parquet reads at this point.
full_years=list(range(1984,2027))
source_rows=[]
for year in full_years:
    p=FIN_DIR/f'YEAR={year}'/'data.parquet'
    assert p.is_file(),f'Missing source: {p}'
    meta=pq.ParquetFile(p).metadata
    source_rows.append({'year':year,'path':str(p),'rows':int(meta.num_rows),'bytes':int(p.stat().st_size)})
source_inventory=pd.DataFrame(source_rows)
assert len(source_inventory)==43
assert int(source_inventory.rows.sum())==TOTAL_EXPECTED_ROWS, (
    'Input row total changed; do not silently compare against the old 03B snapshot.'
)
source_inventory['cumulative_rows']=source_inventory.rows.cumsum()
scales={}
for code,fraction in [('S25',.25),('S50',.50),('S75',.75),('S100',1.00)]:
    target=int(math.ceil(TOTAL_EXPECTED_ROWS*fraction))
    i=int(np.searchsorted(source_inventory.cumulative_rows.to_numpy(),target,side='left'))
    prefix=source_inventory.iloc[:i+1]
    scales[code]={
        'years':prefix.year.astype(int).tolist(),
        'rows':int(prefix.rows.sum()),'files':prefix.path.tolist(),
        'fraction':float(prefix.rows.sum()/TOTAL_EXPECTED_ROWS),
        'bytes':int(prefix.bytes.sum())
    }
active_codes=['S25'] if RUN_MODE=='PILOT' else ['S25','S50','S75','S100']
print('Requested scales:')
for code in active_codes:
    s=scales[code]
    print(code,'years',s['years'][0],'-',s['years'][-1],
          '| rows',f"{s['rows']:,}",f"({s['fraction']:.1%})")
print('NOTE: fractions approximate because whole chronological years are retained.')

Requested scales:
S25 years 1984 - 1990 | rows 482,183 (28.7%)
S50 years 1984 - 1997 | rows 846,510 (50.3%)
S75 years 1984 - 2008 | rows 1,263,644 (75.1%)
S100 years 1984 - 2026 | rows 1,682,616 (100.0%)
NOTE: fractions approximate because whole chronological years are retained.


## 3. Stage local Parquet input (read-only)

Before copying, the notebook checks local free space. Files are staged **only once** for the largest selected scale, and smaller scales refer to prefixes of those files. The source under Google Drive is not modified.

In [ ]:
max_scale=active_codes[-1]
max_years=scales[max_scale]['years']
needed_bytes=sum(int(source_inventory.loc[source_inventory.year==y,'bytes'].iloc[0]) for y in max_years)
usage=shutil.disk_usage(LOCAL_ROOT)
assert needed_bytes < usage.free * LOCAL_STAGE_MAX_DISK_FRACTION, (
    f'Not enough local disk for staging: {needed_bytes/1024**3:.2f} GiB required, '
    f'{usage.free/1024**3:.2f} GiB free.'
)
LOCAL_INPUT=LOCAL_ROOT/'local_parquet'
LOCAL_INPUT.mkdir(parents=True,exist_ok=True)
local_by_year={}
for i,year in enumerate(max_years,1):
    p=Path(source_inventory.loc[source_inventory.year==year,'path'].iloc[0])
    dst=LOCAL_INPUT/f'YEAR={year}'/'data.parquet'
    dst.parent.mkdir(parents=True,exist_ok=True)
    # Match both size and footer row count. Staging files are expendable /content copies.
    expected_rows=int(source_inventory.loc[source_inventory.year==year,'rows'].iloc[0])
    if not (dst.is_file() and dst.stat().st_size==p.stat().st_size
            and pq.ParquetFile(dst).metadata.num_rows==expected_rows):
        shutil.copy2(p,dst)
    assert pq.ParquetFile(dst).metadata.num_rows==expected_rows
    local_by_year[year]=str(dst)
    if i==1 or i%5==0 or i==len(max_years):
        print(f'[Stage local Parquet] {i}/{len(max_years)} years',flush=True)

for code in active_codes:
    scales[code]['local_files']=[local_by_year[y] for y in scales[code]['years']]
print('Local input available:',len(local_by_year),'year files')

[Stage local Parquet] 1/43 years
[Stage local Parquet] 5/43 years
[Stage local Parquet] 10/43 years
[Stage local Parquet] 15/43 years
[Stage local Parquet] 20/43 years
[Stage local Parquet] 25/43 years
[Stage local Parquet] 30/43 years
[Stage local Parquet] 35/43 years
[Stage local Parquet] 40/43 years
[Stage local Parquet] 43/43 years
Local input available: 43 year files


## 4. Benchmark warm-up, repeats and process-isolated memory sampling

**Do not run cells below before the synthetic edge checks PASS.**

Timing includes reading staged local Parquet + processing + materializing a small result. Library interpreter startup and Spark session creation are excluded because warm-up precedes measurement. Each workload explicitly executes after warm-up. RSS is sampled every ~40 ms inside the worker. The `delta_peak_rss_mib` column describes additional observed RSS beyond pre-run baseline, but allocators may retain memory after warm-up, so use caution in comparisons.

The workers write result JSON immediately after each scale. If Colab disconnects midway, earlier output files under `outputs/part3/stage2_benchmark_v1/` remain on Drive.

In [ ]:
import subprocess

def invoke_worker(config):
    outdir=Path(config['outdir'])
    outdir.mkdir(parents=True,exist_ok=True)
    cfgfile=outdir/f"config_{config['engine']}.json"
    cfgfile.write_text(json.dumps(config,indent=2),encoding='utf-8')
    env=os.environ.copy()
    env['POLARS_MAX_THREADS']='2'
    env['OMP_NUM_THREADS']='2'
    env['OPENBLAS_NUM_THREADS']='2'
    cmd=[sys.executable, str(WORKER_PATH),'--config',str(cfgfile)]
    logpath=outdir/f"worker_{config['engine']}.log"
    print(f"Starting {config['engine']} | years {config['years'][0]}-{config['years'][-1]}",flush=True)
    with logpath.open('w',encoding='utf-8') as logfile:
        proc=subprocess.Popen(cmd,env=env,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            print(line.rstrip(),flush=True)
            logfile.write(line)
            logfile.flush()
        rc=proc.wait()
    if rc:
        raise RuntimeError(f"{config['engine']} worker exited {rc}; inspect {logpath}")
    p=outdir/f"{config['engine']}_result.json"
    assert p.exists(),f'Worker did not create output: {p}'
    return json.loads(p.read_text(encoding='utf-8'))

all_worker_results={}
for code in active_codes:
    s=scales[code]
    scale_dir=REPORT_DIR/code
    scale_dir.mkdir(parents=True,exist_ok=True)
    engines=['polars','pandas']
    if RUN_SPARK_BENCHMARK and (RUN_MODE=='PILOT' or code in SPARK_FULL_SCALES):
        engines.append('spark')
    results={}
    for engine in engines:
        cfg={
            'engine':engine,
            'files':s['local_files'],
            'years':s['years'],
            'repeats':(SPARK_PILOT_REPEATS if RUN_MODE=='PILOT'
                       else SPARK_FULL_REPEATS) if engine=='spark' else (
                       PILOT_REPEATS if RUN_MODE=='PILOT' else FULL_REPEATS),
            'outdir':str(scale_dir),
            'workloads':list(logic.WORKLOADS),
        }
        results[engine]=invoke_worker(cfg)
    all_worker_results[code]=results
    # Durable checkpoint after EACH scale.
    checkpoint={"scale":code,"rows":s['rows'],"engines":list(results),
                "completed_utc":datetime.now(timezone.utc).isoformat()}
    (REPORT_DIR/f'stage2_{code}_checkpoint.json').write_text(
        json.dumps(checkpoint,indent=2),encoding='utf-8')
    print('Finished scale:',code,flush=True)

Starting polars | years 1984-1990
Worker polars | 482,183 rows | 7 files
  [polars] W1_FILTER: warm-up
  [polars] W1_FILTER repeat 1/5 5.2 ms | RSS peak 148.5 MiB
  [polars] W1_FILTER repeat 2/5 5.6 ms | RSS peak 148.8 MiB
  [polars] W1_FILTER repeat 3/5 10.1 ms | RSS peak 151.1 MiB
  [polars] W1_FILTER repeat 4/5 8.8 ms | RSS peak 151.4 MiB
  [polars] W1_FILTER repeat 5/5 4.8 ms | RSS peak 151.5 MiB
  [polars] W2_CLEAN_QA: warm-up
  [polars] W2_CLEAN_QA repeat 1/5 141.2 ms | RSS peak 241.9 MiB
  [polars] W2_CLEAN_QA repeat 2/5 135.9 ms | RSS peak 243.6 MiB
  [polars] W2_CLEAN_QA repeat 3/5 125.0 ms | RSS peak 251.6 MiB
  [polars] W2_CLEAN_QA repeat 4/5 172.4 ms | RSS peak 257.8 MiB
  [polars] W2_CLEAN_QA repeat 5/5 123.5 ms | RSS peak 263.3 MiB
  [polars] W3_END_TO_END: warm-up
  [polars] W3_END_TO_END repeat 1/5 386.1 ms | RSS peak 339.5 MiB
  [polars] W3_END_TO_END repeat 2/5 390.7 ms | RSS peak 353.2 MiB
  [polars] W3_END_TO_END repeat 3/5 365.6 ms | RSS peak 361.5 MiB
  [polars] W

## 5. Strict output parity on every executed scale

W1 and W2 results are compact aggregate rows; W3 is a quarter-level table. Compare against Polars, independent of output order. **Counts and identifiers compare exactly**; financial sums use `atol=1e-3, rtol=1e-8` to allow floating-point accumulation order differences. The tolerance is not silently widened.

When Spark is run only at representative scales, mark the remaining Spark comparisons `NOT_RUN` rather than `PASS`.

In [ ]:
PARITY_ROWS=[]

def compare_small_tables(a_records,b_records,scale,workload,engine):
    a=pd.DataFrame(a_records);b=pd.DataFrame(b_records)
    assert set(a.columns)==set(b.columns),(scale,workload,engine,'column mismatch')
    cols=sorted(a.columns)
    if 'REPDTE' in cols:
        a=a.sort_values('REPDTE').reset_index(drop=True)
        b=b.sort_values('REPDTE').reset_index(drop=True)
    else:
        a=a.reset_index(drop=True);b=b.reset_index(drop=True)
    rows_equal=len(a)==len(b)
    PARITY_ROWS.append({'scale':scale,'workload':workload,'engine':engine,
                        'field':'__row_count__','status':'PASS' if rows_equal else 'FAIL',
                        'mismatch_count':abs(len(a)-len(b))})
    if not rows_equal:
        return
    for c in cols:
        if c=='REPDTE':
            bad=(a[c].astype('string').fillna('<NULL>')!=b[c].astype('string').fillna('<NULL>')).sum()
        else:
            x=pd.to_numeric(a[c],errors='coerce').to_numpy(dtype='float64',na_value=np.nan)
            y=pd.to_numeric(b[c],errors='coerce').to_numpy(dtype='float64',na_value=np.nan)
            exact=(c in ['ROWS','BANK_COUNT','NEG_BALANCE_FLAGS','REPORTING_GAP_BANKS']
                   or c.startswith('QA_') or c.endswith('_NULLS'))
            bad=int((~np.isclose(x,y,equal_nan=True,
                   atol=0 if exact else 1e-3,rtol=0 if exact else 1e-8)).sum())
        PARITY_ROWS.append({'scale':scale,'workload':workload,'engine':engine,
                            'field':c,'status':'PASS' if bad==0 else 'FAIL',
                            'mismatch_count':int(bad)})

for code,results in all_worker_results.items():
    for eng in ['pandas','spark']:
        if eng not in results: continue
        for work in logic.WORKLOADS:
            compare_small_tables(results['polars']['results'][work],
                                 results[eng]['results'][work],
                                 code,work,eng)

parity_df=pd.DataFrame(PARITY_ROWS)
parity_df.to_csv(REPORT_DIR/'stage2_parity_results.csv',index=False)
print(parity_df.groupby(['scale','engine','status']).size().to_string())
assert not parity_df.empty
assert (parity_df.status=='PASS').all(), 'Stage 2 parity FAIL; inspect stage2_parity_results.csv.'
print('All executed cross-engine comparisons PASS:',len(parity_df))

scale  engine  status
S100   pandas  PASS      40
       spark   PASS      40
S25    pandas  PASS      40
       spark   PASS      40
S50    pandas  PASS      40
S75    pandas  PASS      40
All executed cross-engine comparisons PASS: 240


## 6. Measured speedup, memory and scale trends

`speedup = pandas median elapsed / Polars median elapsed`. Larger than 1 means Polars faster. RSS is process-tree RSS (not Python heap), and Spark includes JVM; use Spark primarily for plans/local-engine comparison rather than claiming cluster speedup.

In [ ]:
meas=[]
for code,results in all_worker_results.items():
    for eng,info in results.items():
        for row in info['measurements']:
            meas.append({'scale':code,**row})
measurements=pd.DataFrame(meas)
summary=(measurements.groupby(['scale','engine','workload'],as_index=False)
         .agg(median_elapsed_ms=('elapsed_ms','median'),
              median_peak_rss_mib=('peak_rss_mib','median'),
              median_delta_rss_mib=('delta_peak_rss_mib','median'),
              repeats=('repeat','count'),
              input_rows=('rows_input','first')))
wide=summary.pivot_table(index=['scale','workload'],columns='engine',
                         values='median_elapsed_ms',aggfunc='first').reset_index()
if {'pandas','polars'} <= set(wide.columns):
    wide['pandas_over_polars_speedup']=wide['pandas']/wide['polars']
measurements.to_csv(REPORT_DIR/'stage2_raw_timings.csv',index=False)
summary.to_csv(REPORT_DIR/'stage2_benchmark_summary.csv',index=False)
wide.to_csv(REPORT_DIR/'stage2_speedup.csv',index=False)
source_inventory.to_csv(REPORT_DIR/'stage2_source_inventory.csv',index=False)
print('BENCHMARK MEDIANS (measured, not estimates):')
print(summary[['scale','engine','workload','input_rows','median_elapsed_ms',
               'median_peak_rss_mib','repeats']].to_string(index=False))
print('\nSPEEDUP:')
print(wide.to_string(index=False))

BENCHMARK MEDIANS (measured, not estimates):
scale engine      workload  input_rows  median_elapsed_ms  median_peak_rss_mib  repeats
 S100 pandas     W1_FILTER     1682616            613.843              249.414        5
 S100 pandas   W2_CLEAN_QA     1682616           1897.966             1242.652        5
 S100 pandas W3_END_TO_END     1682616           4651.299             3112.887        5
 S100 polars     W1_FILTER     1682616             12.834              147.871        5
 S100 polars   W2_CLEAN_QA     1682616            368.308              458.211        5
 S100 polars W3_END_TO_END     1682616           1459.988              725.367        5
 S100  spark     W1_FILTER     1682616           2391.476              758.676        3
 S100  spark   W2_CLEAN_QA     1682616           6023.092              829.477        3
 S100  spark W3_END_TO_END     1682616           9495.758             1168.871        3
  S25 pandas     W1_FILTER      482183            720.782              162.

## 7. Query plan evidence & cluster justification

Inspect the **actual** plan output and timings. Do not claim streaming unless the engine/plan explicitly verifies it. Spark `Exchange` and `Sort` reflect shuffle and ordering costs; a `BroadcastHashJoin` uses the small calendar dimension. Spark local on 2 CPU cores is **not evidence** that a multi-node cluster is faster.

Cluster discussion should address: observed peak RSS vs VM RAM, elapsed time scaling with rows, workload concurrency, future data growth, network/shuffle overhead and operational overhead. If Polars fits comfortably and meets runtime expectations, the evidence may support **no cluster for the current research dataset**. No universal numerical cutoff is prescribed.

In [ ]:
from pathlib import Path
plan_index=[]
for code,results in all_worker_results.items():
    for eng in results:
        d=REPORT_DIR/code
        for work in logic.WORKLOADS:
            p=d/f'{eng}_{work}_plan.txt'
            if p.exists():
                plan_index.append({'scale':code,'engine':eng,'workload':work,
                                   'file':str(p.relative_to(REPORT_DIR)),'bytes':p.stat().st_size})
plan_df=pd.DataFrame(plan_index)
plan_df.to_csv(REPORT_DIR/'stage2_execution_plan_index.csv',index=False)
print('Execution plan files:',len(plan_df))
if len(plan_df):
    print(plan_df.head(8).to_string(index=False))
    first=REPORT_DIR/plan_df.iloc[0]['file']
    print('\nEXAMPLE plan excerpt:\n',first.read_text(encoding='utf-8')[:2600])

Execution plan files: 18
scale engine      workload                              file  bytes
  S25 polars     W1_FILTER     S25/polars_W1_FILTER_plan.txt    317
  S25 polars   W2_CLEAN_QA   S25/polars_W2_CLEAN_QA_plan.txt   6631
  S25 polars W3_END_TO_END S25/polars_W3_END_TO_END_plan.txt   2262
  S25  spark     W1_FILTER      S25/spark_W1_FILTER_plan.txt   1953
  S25  spark   W2_CLEAN_QA    S25/spark_W2_CLEAN_QA_plan.txt  12363
  S25  spark W3_END_TO_END  S25/spark_W3_END_TO_END_plan.txt   6892
  S50 polars     W1_FILTER     S50/polars_W1_FILTER_plan.txt    319
  S50 polars   W2_CLEAN_QA   S50/polars_W2_CLEAN_QA_plan.txt   6633

EXAMPLE plan excerpt:
 SELECT [len().alias("ROWS"), col("ASSET").sum().alias("ASSET_TOTAL")]
  simple π 1/1 ["ASSET"]
    Parquet SCAN [/content/fdic_part3_stage2/local_parquet/YEAR=1984/data.parquet, ... 6 other sources]
    PROJECT 2/47 COLUMNS
    SELECTION: col("REPDTE").is_between(["19900101", "19901231"])
    ESTIMATED ROWS: 502320


## 8. Acceptance, reproducibility manifest and ZIP output

Only measured results are reported. It is **not** a pass condition that Polars must be faster than pandas; results may vary by workload. Spark not run on intermediate FULL scales is disclosed as `NOT_RUN`.

In [ ]:
gates={
    'STAGE1_CORRECTNESS_REFERENCE_PASS':bool((checks.status=='PASS').all()),
    'EDGE_CASES_THREE_ENGINES_PASS':bool(TEST_SPARK_EDGE and (edge_df.status=='PASS').all()),
    'STAGED_INPUT_COUNTS_MATCH':all(
        sum(pq.ParquetFile(p).metadata.num_rows for p in scales[c]['local_files'])==scales[c]['rows']
        for c in active_codes),
    'POLARS_AND_PANDAS_COMPLETED_ALL_SCALES':all(
        {'polars','pandas'} <= set(all_worker_results[c]) for c in active_codes),
    'EVERY_EXECUTED_PARITY_COMPARISON_PASS':bool((parity_df.status=='PASS').all()),
    'REPEATED_TIMINGS_COMPLETE':bool(len(measurements)>0 and measurements.elapsed_ms.gt(0).all()),
    'SPARK_RUN_WHERE_REQUESTED':bool(not RUN_SPARK_BENCHMARK or all(
        'spark' in all_worker_results[c] for c in active_codes
        if RUN_MODE=='PILOT' or c in SPARK_FULL_SCALES)),
    'FULL_100PCT_COMPLETED_IF_REQUESTED':bool(RUN_MODE!='FULL' or 'S100' in all_worker_results),
}
gate_df=pd.DataFrame([{'gate':k,'status':'PASS' if v else 'FAIL'} for k,v in gates.items()])
gate_df.to_csv(REPORT_DIR/'stage2_acceptance_gates.csv',index=False)
edge_df.to_csv(REPORT_DIR/'stage2_edge_case_checks.csv',index=False)

metadata={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'run_mode':RUN_MODE,'scales':{c:{k:v for k,v in scales[c].items()
                 if k in ('years','rows','fraction','bytes')} for c in active_codes},
    'source_total_rows':TOTAL_EXPECTED_ROWS,
    'software':{'python':sys.version.split()[0],'pandas':pd.__version__,
                'polars':pl.__version__,'pyarrow':pa.__version__,
                'pyspark':__import__('pyspark').__version__,
                'psutil':psutil.__version__},
    'cpu_logical':psutil.cpu_count(logical=True),
    'memory_gib':round(psutil.virtual_memory().total/1024**3,3),
    'platform':platform.platform(),
    'machine':'Colab local VM / local[2] Spark: NOT multiple nodes',
    'rss_method':'Sampled resident memory worker + children, 40ms interval; warm-up excluded',
    'benchmark_scope':'W1 filter/projection; W2 clean/QA with 13 masked sums; W3 end-to-end join/window/aggregation',
    'data_source':'Part1 immutable financials by year (local staging, same files for each engine)',
    'time_method':'wall clock ms, repeat median; warm-up 1 before each workload',
    'not_benchmarked':'Cloud remote I/O, true Spark cluster, production deployment, concurrent users',
    'source_files_not_modified':True,
    'all_acceptance_pass':bool((gate_df.status=='PASS').all()),
    'spark_skipped_scales':[
        c for c in active_codes if 'spark' not in all_worker_results[c]
    ]
}
(REPORT_DIR/'stage2_summary.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
if RUN_MODE=='PILOT' and metadata['all_acceptance_pass']:
    (LOCAL_ROOT/'stage2_pilot_verified.json').write_text(json.dumps(metadata),encoding='utf-8')

zip_path=REPORT_DIR/'FDIC_Part3_Stage2_Benchmark_Reports.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(REPORT_DIR.rglob('*')):
        if p.is_file() and p != zip_path:
            zf.write(p,arcname=str(p.relative_to(REPORT_DIR)))
print('\nSTAGE 2 ACCEPTANCE:')
print(gate_df.to_string(index=False))
print('REPORT ZIP:',zip_path)
assert metadata['all_acceptance_pass'], 'Stage 2 did not pass all gates. Inspect reports.'


STAGE 2 ACCEPTANCE:
                                  gate status
     STAGE1_CORRECTNESS_REFERENCE_PASS   PASS
         EDGE_CASES_THREE_ENGINES_PASS   PASS
             STAGED_INPUT_COUNTS_MATCH   PASS
POLARS_AND_PANDAS_COMPLETED_ALL_SCALES   PASS
 EVERY_EXECUTED_PARITY_COMPARISON_PASS   PASS
             REPEATED_TIMINGS_COMPLETE   PASS
             SPARK_RUN_WHERE_REQUESTED   PASS
    FULL_100PCT_COMPLETED_IF_REQUESTED   PASS
REPORT ZIP: /content/drive/MyDrive/FDIC_Bank_Failure_Project/outputs/part3/stage2_benchmark_v1/full/FDIC_Part3_Stage2_Benchmark_Reports.zip


## 9. How to interpret and advance

**Start now:** `RUN_MODE='PILOT'`, `CONFIRM_FULL=False`; **Run all**. Inspect `stage2_edge_case_checks.csv`, `stage2_parity_results.csv` and `stage2_speedup.csv`. Send the ZIP from `outputs/part3/stage2_benchmark_v1/pilot/` for review.

**Later, deliberately:** change `RUN_MODE='FULL'`, `CONFIRM_FULL=True` and run again. If Colab runtime has restarted, FULL validates the saved PILOT acceptance CSV on Drive; you do not need the in-memory checkpoint. FULL copies every year once but reuses the local stage when valid.

For the Final Report, write three separate claims: correctness (parity + edge cases), performance (measured medians/RSS), and infrastructure decision (current single-VM capacity vs future cluster needs). Do not infer a cluster advantage from PySpark `local[2]` timings.

**Not included here**: a final feature table, label construction, or global imputation; those belong to Part 4. The original 47 financial columns remain unchanged on Drive.